# EDA 보강 — 차트 교체·다변량 분석·관외출동제안발표 피드백(2026-09-28 / 2026-10-02) 대응. 상세 배경은 `docs/progress/제안발표/02_피드백.md`,실행 결과 요약은 `docs/progress/제안발표/03_EDA_보강.md` 참고.**이 노트북이 만드는 것**: `eda/12_*.png` ~ `eda/20_*.png` (9개), `work/ref/eda_multivar.parquet`**핵심 방침**- 기존 5분위 막대그래프는 "구간을 임의로 나눠 생긴 착시 아니냐"는 반박에 약함 →  **연속값 + 20분위 세분화 + 95% Wilson 신뢰구간 + Spearman 상관계수**로 교체- 효과가 다른 변수의 그림자인지 확인하려면 그 변수를 **고정(층화)** 하고도 효과가 남는지 본다

In [ ]:
from pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom scipy.stats import chi2_contingency, pointbiserialr, spearmanrfrom sklearn.linear_model import LinearRegressionROOT = Path().resolve()while not (ROOT / "CLAUDE.md").exists():    ROOT = ROOT.parentplt.rcParams["font.family"] = "Malgun Gothic"plt.rcParams["axes.unicode_minus"] = FalseEDA = ROOT / "eda"WORK_REF = ROOT / "work" / "ref"WORK_REF.mkdir(parents=True, exist_ok=True)

In [ ]:
def wilson_ci(k, n, z=1.96):    """이항비율의 Wilson 95% 신뢰구간. 표본이 작거나 비율이 극단일 때도 안정적이다."""    k = np.asarray(k, dtype=float)    n = np.asarray(n, dtype=float)    p = k / n    denom = 1 + z**2 / n    center = (p + z**2 / (2 * n)) / denom    half = (z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)) / denom    return center - half, center + halfdef mk(dcol, tcol, src):    """일자(YYYYMMDD) + 시각(6자리로 0 채움)을 datetime으로 합친다."""    dd = src[dcol].astype(str).str.strip()    tt = src[tcol].astype(str).str.strip().str.zfill(6)    return pd.to_datetime(dd + tt, format="%Y%m%d%H%M%S", errors="coerce")def rolling_count(df, keycol, timecol, newcol):    """키별 직전 60분 건수. closed='left'라서 자기 자신은 제외된다(누출 방지)."""    d = df.dropna(subset=[timecol]).sort_values(timecol).reset_index(drop=True)    d["one"] = 1    parts = []    for _, g in d.groupby(keycol, sort=False):        g = g.sort_values(timecol).copy()        g[newcol] = g.set_index(timecol)["one"].rolling("60min", closed="left").sum().fillna(0).values        parts.append(g[[keycol, timecol, newcol]])    return pd.concat(parts, ignore_index=True).sort_values(timecol)def binned_rate(df, xcol, ycol, q=20):    """연속형 xcol을 q분위로 나눠 구간별 평균·표본수·Wilson CI를 구한다."""    b = pd.qcut(df[xcol], q, duplicates="drop")    g = df.groupby(b, observed=True).agg(        x=(xcol, "mean"), rate=(ycol, "mean"), n=(ycol, "size"), k=(ycol, "sum")    ).reset_index(drop=True)    lo, hi = wilson_ci(g["k"], g["n"])    g["lo"] = lo * 100    g["hi"] = hi * 100    g["rate_pct"] = g["rate"] * 100    return g

## 1. 데이터 로딩- **활동량은 정제 전 전체 기록(`seoul_2024_raw.csv`)에서 계산**한다 (규칙 4-2). 정제 후 코호트에서 세면  취소·결측 건이 빠져 실제 부하가 과소평가된다.- **라벨 평가는 정제 후 코호트(`seoul_2024_clean.csv`)** 로 한다.

In [ ]:
raw = pd.read_csv(ROOT / "work" / "seoul_2024_raw.csv", dtype=str, low_memory=False)raw["center_id"] = raw["frstn_nm"].str.strip() + " " + raw["cntr_nm"].str.strip()raw["dclr"] = mk("dclr_ymd", "dclr_tm", raw)raw["dspt"] = mk("dspt_ymd", "dspt_tm", raw)clean = pd.read_csv(ROOT / "work" / "seoul_2024_clean.csv", dtype=str, low_memory=False)for c in ["dclr", "dspt", "arvl"]:    clean[c] = pd.to_datetime(clean[c])clean["late"] = clean["late"].map({"True": 1, "False": 0}).astype(float)clean["hour"] = clean["hour"].astype(int)clean["resp_min"] = clean["resp_min"].astype(float)clean["dist_km"] = pd.to_numeric(clean["dist_km"], errors="coerce")print(f"raw(전체 기록): {len(raw):,}행 / clean(라벨 코호트): {len(clean):,}행")

## 2. 응답시간 CDF와 임계값 민감도 → `eda/12`기존 히스토그램(`02_resp_dist.png`)을 대체한다. 타깃이 "8분 내 도착 확률"이므로 **누적분포(CDF)의 8분 지점이곧 타깃**이다. 같은 그림에 7·9·10분을 함께 표시해 **"8분이라는 임계값을 어디 두느냐에 따라 결론이 달라지는가"**(2차 피드백)에 답한다.

In [ ]:
resp = np.sort(clean["resp_min"].values)cdf = np.arange(1, len(resp) + 1) / len(resp)fig, ax = plt.subplots(figsize=(8, 5))ax.plot(resp, cdf * 100, color="#2563eb", lw=1.5)ax.set_xlim(0, 20)ax.set_xlabel("신고~현장도착 소요시간(분)")ax.set_ylabel("누적 비율(%)")ax.set_title("서울 2024 — 응답시간 누적분포(CDF)와 임계값별 초과율")ax.grid(alpha=0.3)for t, c in zip([7, 8, 9, 10], ["#f59e0b", "#dc2626", "#f59e0b", "#f59e0b"]):    pct_under = (resp <= t).mean() * 100    ax.axvline(t, color=c, lw=2.2 if t == 8 else 1.2, ls="-" if t == 8 else "--", alpha=0.8)    ax.annotate(f"{t}분: 초과율 {100 - pct_under:.1f}%", xy=(t, pct_under),                xytext=(t + 0.3, pct_under - (15 if t == 8 else 8)), fontsize=9, color=c)p90 = np.percentile(resp, 90)ax.axhline(90, color="gray", lw=0.8, ls=":")ax.annotate(f"90퍼센타일 = {p90:.2f}분", xy=(p90, 90), xytext=(p90 + 0.5, 82), fontsize=9, color="gray")plt.tight_layout()plt.savefig(EDA / "12_cdf_threshold.png", dpi=150)plt.show()for t in [7, 8, 9, 10]:    print(f"{t}분 초과율: {(resp > t).mean() * 100:.2f}%")print(f"평균 {resp.mean():.2f}분 / 중앙값 {np.median(resp):.2f}분 / 90퍼센타일 {p90:.2f}분")

## 3. 통합 분석 테이블 → `work/ref/eda_multivar.parquet`호출(call) 1건 단위로 활동량·정체·거리·시간을 모두 붙인다. 이후 모든 다변량 분석이 이 테이블 하나를 쓴다.| 변수 | 기준 시각 | 비고 || --- | --- | --- || `sgg_load` | **신고시각(dclr)** | 구별 직전 60분 신고 건수 || `center_load` | **출동시각(dspt)** | 센터별 직전 60분 출동 건수 (규칙상 센터 활동량은 출동시각 기준) || `avg_speed_kmh` | 신고 **직전 완결 시간** | `speed_by_gu_hour`의 `hour=h`는 `[h-1시, h시)` 구간. `hour=0`이면 전날 24시 코드 |

In [ ]:
gu_ev = rolling_count(raw, "sgg_nm", "dclr", "sgg_load")cid_ev = rolling_count(raw, "center_id", "dspt", "center_load")base = clean[["rlf_rptp_no", "sgg_nm", "center_id", "dclr", "dspt", "arvl",              "hour", "dclr_dow", "dist_km", "resp_min", "late"]].copy()m = pd.merge_asof(base.sort_values("dclr"), gu_ev, on="dclr", by="sgg_nm",                  direction="backward", tolerance=pd.Timedelta("60min"))m["sgg_load"] = m["sgg_load"].fillna(0)m = m.dropna(subset=["dspt", "center_id"]).sort_values("dspt")m = pd.merge_asof(m, cid_ev, on="dspt", by="center_id",                  direction="backward", tolerance=pd.Timedelta("60min"))m["center_load"] = m["center_load"].fillna(0)# 직전 완결 시간으로 속도 결합ymd_int = m["dclr"].dt.strftime("%Y%m%d").astype(int)prev_day = (m["dclr"] - pd.Timedelta(days=1)).dt.strftime("%Y%m%d").astype(int)m["join_ymd"] = np.where(m["hour"] != 0, ymd_int, prev_day)m["join_hour"] = np.where(m["hour"] != 0, m["hour"], 24)speed = pd.read_parquet(ROOT / "work" / "speed" / "speed_by_gu_hour.parquet")scols = speed.columns.tolist()  # ['일자','요일','sgg_nm','hour','avg_speed_kmh']speed = speed.rename(columns={scols[0]: "join_ymd", scols[3]: "join_hour"})speed_2024 = speed[speed["join_ymd"].between(20240101, 20241231)]final = m.merge(speed_2024[["join_ymd", "sgg_nm", "join_hour", "avg_speed_kmh"]],                on=["join_ymd", "sgg_nm", "join_hour"], how="left")final.to_parquet(WORK_REF / "eda_multivar.parquet", index=False)print(f"{len(final):,}행")print("결측률:", (final[["sgg_load", "center_load", "avg_speed_kmh", "dist_km"]].isna().mean() * 100).round(3).to_dict())

### 3-1. 거리 이상치 확인`dist_km`(대원 기재 정수 km)에 **음수와 비현실적 값**이 섞여 있다. 상관분석을 왜곡하므로 제외한다.6km 초과(상위 1%, `out_of_range` 기준)는 **오류가 아니므로 남긴다**.

In [ ]:
print(f"음수: {(final['dist_km'] < 0).sum()}건")print(f"100km 초과(명백한 오류): {(final['dist_km'] > 100).sum()}건")print("이상값 목록:", sorted(final.loc[final["dist_km"] > 100, "dist_km"].unique()))print(sorted(final.loc[final["dist_km"] < 0, "dist_km"].unique()))ok = final["dist_km"].isna() | final["dist_km"].between(0, 100)d = final[ok].copy()print(f"\n제외 후: {len(d):,}행 (제외 {len(final) - len(d)}건)")

## 4. RQ1-a 구 활동량 → `eda/13`"신고 지역이 바쁘면 늦어지는가". 기존 5분위 막대(`06_sgg_load.png`)를 대체한다.구간화 전 **연속값 상관계수**를 함께 제시해 "구간을 어떻게 나누느냐에 따라 달라진다"는 반박을 막는다.

In [ ]:
g = binned_rate(d.dropna(subset=["sgg_load"]), "sgg_load", "late", q=20)g60 = binned_rate(d.dropna(subset=["sgg_load"]), "sgg_load", "late", q=60)rho, p = spearmanr(d["sgg_load"], d["late"])fig, ax = plt.subplots(figsize=(8, 5.5))ax.errorbar(g["x"], g["rate_pct"], yerr=[g["rate_pct"] - g["lo"], g["hi"] - g["rate_pct"]],            fmt="o", color="#2563eb", ecolor="#93c5fd", capsize=3, markersize=6, zorder=3,            label="20분위 구간별 평균 ± 95% CI(Wilson)")ax.plot(g60["x"], g60["rate_pct"], color="#dc2626", lw=1.8, alpha=0.8, zorder=2, label="60분위 추세")ax.axhline(d["late"].mean() * 100, color="gray", ls="--", lw=1, label="전체 평균")ax.set_xlabel("신고 직전 60분간 해당 구 신고 건수 (연속값)")ax.set_ylabel("8분 초과율 (%)")ax.set_title(f"구 활동량과 8분 초과율 — 연속값 기준 (Spearman ρ={rho:.3f}, p<0.001)")ax.legend(fontsize=9)ax.grid(alpha=0.3)plt.tight_layout()plt.savefig(EDA / "13_sgg_load_continuous.png", dpi=150)plt.show()print(f"Spearman ρ={rho:.4f}, p={p:.2e}")print(g[["x", "rate_pct", "n"]].round(2).to_string())

## 5. RQ1-b 센터 활동량 → `eda/14`"출동한 센터가 바쁘면 늦어지는가". 결과는 **효과 없음**이다.x축 범위가 0~4건으로 좁은 것 자체도 의미가 있다 — 한 센터가 60분에 4건 넘게 출동하는 경우가 드물어"센터가 바쁘다"는 신호 자체가 약하다. 관할 센터가 바쁘면 **인접 센터가 출동**해 기록에 남지 않기 때문이다.

In [ ]:
rho_c, p_c = spearmanr(d["center_load"], d["late"])r_pb, p_pb = pointbiserialr(d["late"], d["center_load"])g = binned_rate(d, "center_load", "late", q=20)g60 = binned_rate(d, "center_load", "late", q=60)fig, ax = plt.subplots(figsize=(8, 5.5))ax.errorbar(g["x"], g["rate_pct"], yerr=[g["rate_pct"] - g["lo"], g["hi"] - g["rate_pct"]],            fmt="o", color="#16a34a", ecolor="#86efac", capsize=3, markersize=6, zorder=3,            label="20분위 구간별 평균 ± 95% CI(Wilson)")ax.plot(g60["x"], g60["rate_pct"], color="#15803d", lw=1.5, alpha=0.7, zorder=2, label="60분위 추세")ax.axhline(d["late"].mean() * 100, color="gray", ls="--", lw=1, label="전체 평균")ax.set_ylim(40, 56)ax.set_xlabel("출동 직전 60분간 해당 센터 출동 건수 (연속값)")ax.set_ylabel("8분 초과율 (%)")ax.set_title(f"센터 활동량과 8분 초과율 — 연속값 기준 (Spearman ρ={rho_c:.3f}, n.s.)")ax.legend(fontsize=9)ax.grid(alpha=0.3)plt.tight_layout()plt.savefig(EDA / "14_center_load_continuous.png", dpi=150)plt.show()print(f"Spearman ρ={rho_c:.4f} (p={p_c:.2e}) / point-biserial r={r_pb:.4f} (p={p_pb:.3f})")

## 6. RQ2 교통 정체 → `eda/15`"도로가 막히면 늦어지는가". 방향은 가설대로지만 **가장 원활한 구간(30km/h 이상)에서 소폭 반등**해완전한 단조감소가 아니다. 5분위로만 보면 이 반등이 Q5 안에 묻혀 보이지 않았다. 숨기지 않고 기록한다.

In [ ]:
dv = d.dropna(subset=["avg_speed_kmh"])rho_s, p_s = spearmanr(dv["avg_speed_kmh"], dv["late"])g = binned_rate(dv, "avg_speed_kmh", "late", q=20)g60 = binned_rate(dv, "avg_speed_kmh", "late", q=60)fig, ax = plt.subplots(figsize=(8, 5.5))ax.errorbar(g["x"], g["rate_pct"], yerr=[g["rate_pct"] - g["lo"], g["hi"] - g["rate_pct"]],            fmt="o", color="#ea580c", ecolor="#fed7aa", capsize=3, markersize=6, zorder=3,            label="20분위 구간별 평균 ± 95% CI(Wilson)")ax.plot(g60["x"], g60["rate_pct"], color="#c2410c", lw=1.5, alpha=0.7, zorder=2, label="60분위 추세")ax.axhline(dv["late"].mean() * 100, color="gray", ls="--", lw=1, label="전체 평균")ax.set_xlabel("신고 직전 완결 시간의 구 평균 속도 (km/h) — 왼쪽=정체, 오른쪽=원활")ax.set_ylabel("8분 초과율 (%)")ax.set_title(f"구 평균 속도와 8분 초과율 — 연속값 기준 (Spearman ρ={rho_s:.3f}, p<0.001)")ax.legend(fontsize=9)ax.grid(alpha=0.3)plt.tight_layout()plt.savefig(EDA / "15_speed_cong_continuous.png", dpi=150)plt.show()print(f"Spearman ρ={rho_s:.4f}, p={p_s:.2e}")

## 7. 상관행렬과 VIF교수님 피드백 **"차트들의 상관관계"** 에 대한 답. 각 효과가 서로 중복인지 독립인지 판단하는 근거다.- **거리-지연 상관(0.403)이 RQ1/RQ2 변수보다 훨씬 크다.** 당연한 물리적 관계지만,  발표에서 "거리는 중요하지 않다"처럼 말하면 이 표와 모순되어 보인다.  → "거리도 중요하지만, 거리로 설명 안 되는 추가 변동이 있다"로 서술해야 한다.- **VIF가 전부 10 미만**이면 다중공선성 문제가 없다는 뜻 → RQ1+RQ2 결합 모델(RQ3)을 진행해도 통계적으로 안전하다.

In [ ]:
cols = ["sgg_load", "center_load", "avg_speed_kmh", "dist_km", "hour", "late"]dc = d[cols].dropna()print(f"n = {len(dc):,}")print("\n=== Spearman 상관행렬 ===")print(dc.corr(method="spearman").round(3).to_string())X_cols = ["sgg_load", "center_load", "avg_speed_kmh", "dist_km", "hour"]print("\n=== VIF (10 이상이면 다중공선성 문제) ===")for c in X_cols:    others = [cc for cc in X_cols if cc != c]    r2 = LinearRegression().fit(dc[others].values, dc[c].values).score(dc[others].values, dc[c].values)    print(f"{c}: {1 / (1 - r2):.3f}")

## 8. 교차 층화표 → `eda/16`, `eda/17`**핵심 질문**: 구 활동량 효과가 사실 정체·시간대 때문에 생긴 착시(교란)인가?검증 방법은 **교란 의심 변수를 고정해 놓고도 효과가 남는지** 보는 것이다.

In [ ]:
dx = d.dropna(subset=["avg_speed_kmh"]).copy()dx["load_q"] = pd.qcut(dx["sgg_load"], 5, labels=["Q1(한산)", "Q2", "Q3", "Q4", "Q5(혼잡)"])dx["speed_q"] = pd.qcut(dx["avg_speed_kmh"], 5, labels=["S1(정체)", "S2", "S3", "S4", "S5(원활)"])dt = d.copy()dt["load_q"] = pd.qcut(dt["sgg_load"], 5, labels=["Q1(한산)", "Q2", "Q3", "Q4", "Q5(혼잡)"])bands = ["심야(0-5)", "오전(6-11)", "오후(12-17)", "저녁(18-23)"]dt["time_band"] = pd.cut(dt["hour"], bins=[-1, 5, 11, 17, 23], labels=bands)def draw_cross(df, colvar, fname, xlabel, title):    rate = df.pivot_table(index="load_q", columns=colvar, values="late",                          aggfunc="mean", observed=True) * 100    n = df.pivot_table(index="load_q", columns=colvar, values="late", aggfunc="size", observed=True)    fig, ax = plt.subplots(figsize=(8, 6))    im = ax.imshow(rate.values, cmap="Reds", aspect="auto", vmin=35, vmax=65)    ax.set_xticks(range(rate.shape[1]), rate.columns)    ax.set_yticks(range(rate.shape[0]), rate.index)    ax.set_xlabel(xlabel)    ax.set_ylabel("구 활동량 5분위 (아래=혼잡)")    ax.set_title(title)    for i in range(rate.shape[0]):        for j in range(rate.shape[1]):            v = rate.values[i, j]            ax.text(j, i, f"{v:.1f}%\n(n={n.values[i, j]:,})", ha="center", va="center",                    fontsize=8, color="white" if v > 52 else "black")    plt.colorbar(im, label="8분 초과율(%)")    plt.tight_layout()    plt.savefig(EDA / fname, dpi=150)    plt.show()    print(f"[{fname}]")    for col in rate.columns:        lo_v, hi_v = rate[col].iloc[0], rate[col].iloc[-1]        print(f"{col}: Q1={lo_v:.1f}% -> Q5={hi_v:.1f}% (차이 {hi_v - lo_v:+.1f}p)")    return rate_ = draw_cross(dx, "speed_q", "16_cross_load_speed.png",               "정체 5분위 (왼쪽=정체 심함)", "정체 × 구활동량 교차 — 8분 초과율(%)")_ = draw_cross(dt, "time_band", "17_cross_load_hour.png",               "시간대", "시간대 × 구활동량 교차 — 8분 초과율(%)")

## 9. 층화 추세 + 유의성 검정 → `eda/18`교차표를 통계적으로 확정한다. **정체 구간마다 따로** (연속값 활동량 ↔ 지연) 상관을 구해,모든 구간에서 유의한지 본다. 선의 **기울기 차이**는 상호작용(정체가 심할수록 활동량 효과가 커짐)을 보여준다.

In [ ]:
dx["load_q5"] = pd.qcut(dx["sgg_load"], 5, labels=[1, 2, 3, 4, 5]).astype(int)fig, ax = plt.subplots(figsize=(8, 5.5))for (level, g_), c in zip(dx.groupby("speed_q", observed=True), plt.cm.Reds(np.linspace(0.35, 0.95, 5))):    agg = g_.groupby("load_q5")["late"].agg(["mean", "size", "sum"])    lo, hi = wilson_ci(agg["sum"], agg["size"])    rho_i, p_i = spearmanr(g_["sgg_load"], g_["late"])    ax.errorbar(agg.index, agg["mean"] * 100,                yerr=[agg["mean"] * 100 - lo * 100, hi * 100 - agg["mean"] * 100],                marker="o", capsize=3, color=c, label=f"{level} (ρ={rho_i:.3f}, p<0.001)")    print(f"{level}: ρ={rho_i:.4f}, p={p_i:.2e}, n={len(g_):,}")ax.set_xticks([1, 2, 3, 4, 5], ["Q1(한산)", "Q2", "Q3", "Q4", "Q5(혼잡)"])ax.set_xlabel("구 활동량 5분위")ax.set_ylabel("8분 초과율 (%)")ax.set_title("정체 수준별 — 구활동량 효과의 일관성 (95% CI)")ax.legend(fontsize=8, title="정체 구간(그 안에서의 상관계수)")ax.grid(alpha=0.3)plt.tight_layout()plt.savefig(EDA / "18_stratified_trend_speed.png", dpi=150)plt.show()

## 10. 관외출동 → `eda/19`, `eda/20`1차 피드백 **"구 안 센터가 출동해도 도착지점까지 구 경계를 넘어갈 수 있는데 고려했나"** 에 대한 답.`sgg_nm`은 **신고 위치**의 구이고 센터 소속 구가 아니다. 안전센터 현황(주소)으로 센터 소속 구를 구해 비교한다.매핑 테이블 생성은 별도 작업(`work/ref/center_to_sgg.parquet`).

In [ ]:
ref = pd.read_parquet(WORK_REF / "center_to_sgg.parquet")[["center_id", "center_sgg"]]oj = clean.merge(ref, on="center_id", how="left")print(f"서울 안전센터로 매칭: {oj['center_sgg'].notna().sum():,} / {len(oj):,} "      f"({oj['center_sgg'].notna().mean() * 100:.1f}%)  — 나머지는 서울 밖 소방서·비안전센터")oj = oj.dropna(subset=["center_sgg"]).copy()oj["out_of_jur"] = (oj["sgg_nm"] != oj["center_sgg"]).astype(int)print(f"관외출동 비율: {oj['out_of_jur'].mean() * 100:.2f}%")chi2, p_chi, _, _ = chi2_contingency(pd.crosstab(oj["out_of_jur"], oj["late"]))agg = oj.groupby("out_of_jur")["late"].agg(["mean", "sum", "size"])lo, hi = wilson_ci(agg["sum"], agg["size"])fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))ax = axes[0]bars = ax.bar(["관내(일치)", "관외(불일치)"], agg["mean"] * 100, color=["#2563eb", "#dc2626"],              yerr=[(agg["mean"] * 100 - lo * 100).values, (hi * 100 - agg["mean"] * 100).values], capsize=5)for b, v, n_ in zip(bars, agg["mean"] * 100, agg["size"]):    ax.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%\n(n={n_:,})", ha="center", fontsize=9)ax.set_ylabel("8분 초과율(%)")ax.set_title(f"관내 vs 관외출동 — 8분 초과율\nchi2={chi2:.1f}, p<0.001")ax.set_ylim(0, 80)ax = axes[1]gu_rate = (oj.groupby("sgg_nm")["out_of_jur"].mean() * 100).sort_values()ax.barh(gu_rate.index, gu_rate.values, color="#f59e0b")ax.axvline(oj["out_of_jur"].mean() * 100, color="gray", ls="--", label="전체 평균")ax.set_xlabel("관외출동 비율(%)")ax.set_title("구별 관외출동 비율")ax.legend(fontsize=8)plt.tight_layout()plt.savefig(EDA / "19_out_of_jurisdiction.png", dpi=150)plt.show()print(f"관내 95%CI: {lo[0] * 100:.2f}~{hi[0] * 100:.2f} / "      f"관외 95%CI: {lo[1] * 100:.2f}~{hi[1] * 100:.2f}")

### 10-1. "그냥 거리가 멀어서 아니냐"에 대한 답 → `eda/20`관외출동은 거리와 겹친다(상관 r≈0.28, 평균거리 2.51km vs 1.69km). 교수님 지적대로 거리가 일부 설명한다.그러나 **거리를 5분위로 고정해도 관외 효과가 그대로 남으면**, 거리만으로는 전부 설명되지 않는다는 뜻이다.

In [ ]:
ojd = oj[oj["dist_km"].isna() | oj["dist_km"].between(0, 100)].dropna(subset=["dist_km"]).copy()r_d, _ = pointbiserialr(ojd["out_of_jur"], ojd["dist_km"])print(f"out_of_jur <-> dist_km 상관 r={r_d:.4f}")print(ojd.groupby("out_of_jur")["dist_km"].mean().round(3).to_string())ojd["dist_q"] = pd.qcut(ojd["dist_km"], 5, labels=["D1(최단)", "D2", "D3", "D4", "D5(최장)"])rows = []for dq, g_ in ojd.groupby("dist_q", observed=True):    a = g_.groupby("out_of_jur")["late"].mean() * 100    rows.append((dq, a.loc[0], a.loc[1], a.loc[1] - a.loc[0]))res = pd.DataFrame(rows, columns=["거리구간", "관내율", "관외율", "차이"])fig, ax = plt.subplots(figsize=(8, 5.5))x = np.arange(len(res))ax.bar(x - 0.175, res["관내율"], width=0.35, color="#2563eb", label="관내")ax.bar(x + 0.175, res["관외율"], width=0.35, color="#dc2626", label="관외")for i, row in res.iterrows():    ax.text(i, max(row["관내율"], row["관외율"]) + 1, f"+{row['차이']:.1f}p", ha="center", fontsize=9)ax.set_xticks(x, res["거리구간"])ax.set_ylabel("8분 초과율(%)")ax.set_xlabel("거리 5분위")ax.set_title("거리를 통제한 후 — 관내 vs 관외출동 효과")ax.legend()plt.tight_layout()plt.savefig(EDA / "20_outofjur_controlled_by_dist.png", dpi=150)plt.show()print(res.round(2).to_string(index=False))

### 10-2. 남은 효과는 어디서 생기나 — 준비시간 vs 이동시간거리로 설명 안 되는 8~9.6p가 **어느 구간에서** 생기는지 분해한다.- **출동준비시간**(신고→출동): 상황실이 판단하고 차를 내보내기까지 → 차이가 크면 "행정 조율" 가설- **이동시간**(출동→도착): 실제 주행 → 차이가 크면 "낯선 구역이라 비효율" 가설결과는 **이동시간 쪽**이다. 다만 이것이 "진짜 지리 비효율"인지 "`grnds_dstnc` 기록이 관외 건에서 부정확"한지는구분하지 못한다 → **한계로 명시**.

In [ ]:
dec = oj.copy()dec["prep_min"] = (dec["dspt"] - dec["dclr"]).dt.total_seconds() / 60dec["travel_min"] = (dec["arvl"] - dec["dspt"]).dt.total_seconds() / 60dec = dec[dec["prep_min"].between(0, 30) & dec["travel_min"].between(0, 60)]dec = dec[dec["dist_km"].isna() | dec["dist_km"].between(0, 100)].dropna(subset=["dist_km"])dec["dist_q"] = pd.qcut(dec["dist_km"], 5, labels=["D1", "D2", "D3", "D4", "D5"])print("=== 전체 평균 (0=관내, 1=관외) ===")print(dec.groupby("out_of_jur")[["prep_min", "travel_min"]].mean().round(3).to_string())print("\n=== 거리구간별 (관외 - 관내) ===")for dq, g_ in dec.groupby("dist_q", observed=True):    a = g_.groupby("out_of_jur")[["prep_min", "travel_min"]].mean()    print(f"{dq}: 준비시간 {a.loc[1, 'prep_min'] - a.loc[0, 'prep_min']:+.3f}분, "          f"이동시간 {a.loc[1, 'travel_min'] - a.loc[0, 'travel_min']:+.3f}분 (n={len(g_):,})")